# 09 — Testing whether DINOv2 and SatCLIP provide complementary information

DINOv2 summarises local aerial imagery, while SatCLIP represents geographical location and its broader environmental context. This notebook tests whether combining the two improves prediction beyond either representation alone.

Five models are evaluated: the combined representation by itself for PTAL and EPC, and the same pair added to the PTAL location baseline, compact EPC controls and richer EPC controls. The models use the same five held-out borough rounds and Ridge procedure as the main analysis. Existing DINOv2-only, SatCLIP-only and all-representation results provide direct comparators on the same test locations.

## Main findings

For PTAL without controls, DINOv2 plus SatCLIP improves mean R² over DINOv2 alone by about 0.0064, but MAE becomes slightly worse. When the PTAL location baseline is already present, the pair performs about 0.0023 R² below DINOv2 alone and does not improve any metric in any held-out round.

For EPC, the change relative to DINOv2 is approximately +0.00016 without controls, +0.00027 with compact controls and −0.00039 with richer controls. These differences are practically negligible.

The combined result indicates that SatCLIP contributes little information beyond DINOv2 for these tasks. The two-encoder pair is compact, but it does not reproduce the performance of the full representation combination.

In [ ]:
# Connect Google Drive and load packages for the targeted representation combination.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith("_")
})

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

print("Python:", platform.python_version())
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)

required_paths = [
    FINAL_MODEL_TABLE_PATH,
    FEATURE_MANIFEST_JSON_PATH,
    RIDGE_OUTER_FOLDS_PATH,
    RIDGE_RUN_SPEC_PATH,
    RIDGE_CORE_AUDIT_PATH,
    RIDGE_CORE_RESULTS_PATH,
    RIDGE_CORE_PREDICTIONS_PATH,
    INCREMENTAL_RUN_SPEC_PATH,
    INCREMENTAL_AUDIT_PATH,
    INCREMENTAL_RESULTS_PATH,
    INCREMENTAL_PREDICTIONS_PATH,
]
for path in required_paths:
    print(path, path.exists())
    assert path.exists(), f"Missing prerequisite: {path}"

## 1. Load the established analysis definitions

The common table, feature manifest, borough assignments and source model results are read from the earlier analyses. Their identifiers are compared to ensure that the new pair is evaluated under the same data and model conditions.

In [ ]:
# Read the common data, source results and their stored analysis identities.
df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
with open(FEATURE_MANIFEST_JSON_PATH, "r") as f:
    manifest = json.load(f)
with open(RIDGE_RUN_SPEC_PATH, "r") as f:
    source_06_spec = json.load(f)
with open(RIDGE_CORE_AUDIT_PATH, "r") as f:
    source_06_audit = json.load(f)
with open(INCREMENTAL_RUN_SPEC_PATH, "r") as f:
    source_07_spec = json.load(f)
with open(INCREMENTAL_AUDIT_PATH, "r") as f:
    source_07_audit = json.load(f)

assert source_06_audit["integrity_gate_pass"] is True
assert source_06_audit["interpretation_gate_pass"] is True
assert source_07_audit["integrity_gate_pass"] is True
assert source_07_audit["interpretation_gate_pass"] is True

target_col = manifest["target_column"]
group_col = manifest["group_column"]
categorical_master = set(manifest["categorical_columns"])
feature_sets = manifest["feature_sets"]

assert len(df) == 26597
assert df["sample_id"].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert df[group_col].nunique() == 33
assert set(df["task"].unique()) == {"PTAL", "EPC"}

df = df.sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)
task_counts = df["task"].value_counts().to_dict()
assert task_counts == {"EPC": 20000, "PTAL": 6597}

model_key_frame = df[["sample_id", "task", group_col, target_col]].copy()
model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(model_key_frame, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()
source_06_spec_sha256 = hashlib.sha256(
    json.dumps(source_06_spec, sort_keys=True).encode("utf-8")
).hexdigest()
source_07_spec_sha256 = hashlib.sha256(
    json.dumps(source_07_spec, sort_keys=True).encode("utf-8")
).hexdigest()

assert source_06_spec["model_key_sha256"] == model_key_hash
assert source_07_spec["model_key_sha256"] == model_key_hash
assert source_06_spec["feature_manifest_sha256"] == manifest_hash
assert source_07_spec["feature_manifest_sha256"] == manifest_hash
assert source_06_spec["outer_fold_assignment_sha256"] == source_07_spec["outer_fold_assignment_sha256"]
assert int(source_06_spec["outer_splits"]) == int(source_07_spec["outer_splits"]) == int(RIDGE_OUTER_SPLITS) == 5
assert int(source_06_spec["inner_splits"]) == int(source_07_spec["inner_splits"]) == int(RIDGE_INNER_SPLITS) == 3
assert [float(x) for x in source_06_spec["alpha_grid"]] == [float(x) for x in RIDGE_ALPHA_GRID]
assert [float(x) for x in source_07_spec["alpha_grid"]] == [float(x) for x in RIDGE_ALPHA_GRID]

print("Frozen prerequisite identity: PASS")
print("Model-key SHA256:", model_key_hash)
print("Manifest SHA256:", manifest_hash)
display(pd.Series(task_counts, name="n"))

## 2. Define the DINOv2–SatCLIP models and comparators

The new feature set is the ordered union of the existing DINOv2 and SatCLIP columns. For each task and control setting, it is compared with DINOv2 alone, SatCLIP alone and the corresponding all-representation model. The comparison with DINOv2 directly tests whether SatCLIP adds information; the full model shows the performance cost or benefit of using only two encoders.

In [ ]:
# Define the DINOv2–SatCLIP feature union and its task-specific comparators.
for name in [
    "DINOv2", "SatCLIP", "All_representations_plus_SV_metadata",
    "PTAL_spatial_baseline", "EPC_controls_sparse", "EPC_controls_extensive",
]:
    assert name in feature_sets and feature_sets[name], f"Missing frozen feature set: {name}"

def dedupe_preserve_order(columns):
    return list(dict.fromkeys(columns))

def columns_sha256(columns):
    return hashlib.sha256(
        json.dumps(list(columns), separators=(",", ":")).encode("utf-8")
    ).hexdigest()

image_location_cols = dedupe_preserve_order(
    list(feature_sets["DINOv2"]) + list(feature_sets["SatCLIP"])
)
assert len(image_location_cols) == len(feature_sets["DINOv2"]) + len(feature_sets["SatCLIP"])
assert not [c for c in image_location_cols if c not in df.columns]

new_definitions = [
    ("PTAL", "PTAL_representation_only__DINOv2_plus_SatCLIP", None, "representation_only"),
    ("EPC", "EPC_representation_only__DINOv2_plus_SatCLIP", None, "representation_only"),
    ("PTAL", "PTAL_spatial_baseline__plus__DINOv2_plus_SatCLIP", "PTAL_spatial_baseline", "controls_plus"),
    ("EPC", "EPC_controls_sparse__plus__DINOv2_plus_SatCLIP", "EPC_controls_sparse", "controls_plus"),
    ("EPC", "EPC_controls_extensive__plus__DINOv2_plus_SatCLIP", "EPC_controls_extensive", "controls_plus_privileged"),
]

model_rows = []
model_features = {}
for task, model_id, base_set, analysis_role in new_definitions:
    cols = image_location_cols if base_set is None else dedupe_preserve_order(
        list(feature_sets[base_set]) + image_location_cols
    )
    assert cols and not [c for c in cols if c not in df.columns]
    model_features[model_id] = cols
    model_rows.append({
        "task": task,
        "model_id": model_id,
        "base_feature_set": base_set,
        "analysis_role": analysis_role,
        "n_features_manifest": len(cols),
    })

model_specs = pd.DataFrame(model_rows)
assert len(model_specs) == 5 and model_specs["model_id"].is_unique
assert model_specs.groupby("task").size().to_dict() == {"EPC": 3, "PTAL": 2}

comparator_rows = []
for row in model_specs.itertuples(index=False):
    if row.base_feature_set is None:
        source_notebook = "06"
        comparator_ids = {
            "DINOv2_only": "DINOv2",
            "SatCLIP_only": "SatCLIP",
            "full_fusion": "All_representations_plus_SV_metadata",
        }
    else:
        source_notebook = "07"
        comparator_ids = {
            "DINOv2_only": f"{row.base_feature_set}__plus__DINOv2",
            "SatCLIP_only": f"{row.base_feature_set}__plus__SatCLIP",
            "full_fusion": f"{row.base_feature_set}__plus__All_representations_plus_SV_metadata",
        }
    for comparator_role, comparator_id in comparator_ids.items():
        comparator_rows.append({
            "task": row.task,
            "new_model_id": row.model_id,
            "comparator_role": comparator_role,
            "source_notebook": source_notebook,
            "comparator_id": comparator_id,
        })

comparator_specs = pd.DataFrame(comparator_rows)
assert len(comparator_specs) == 15
display(model_specs)
display(comparator_specs)
print("DINOv2 columns:", len(feature_sets["DINOv2"]))
print("SatCLIP columns:", len(feature_sets["SatCLIP"]))
print("Combined columns:", len(image_location_cols))

## 3. Use training-only preprocessing

Missing-value treatment, standardisation, categorical encoding and Ridge penalty selection are learned within each training split. The pretrained representations are used as fixed features; the encoders themselves are not fine-tuned.

In [ ]:
# Build the same training-only preprocessing and Ridge pipeline.
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_pipeline(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []
    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ))
    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", make_onehot()),
            ]),
            categorical_cols,
        ))
    pre = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)
    return Pipeline([
        ("preprocess", pre),
        ("ridge", Ridge(solver="lsqr", max_iter=5000, tol=1e-4)),
    ])

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    mapped = series.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    assert mapped.notna().all(), "Unexpected boolean encoding"
    return mapped.astype(bool)

## 4. Reuse the borough-based test rounds

The saved borough assignments are reconstructed and compared row by row before fitting. Existing partial results are accepted only when their model definition and held-out sample IDs match the current analysis.

In [ ]:
# Reconstruct the established borough test rounds and record the new model specification.
outer_assignment_rows = []
fold_index_by_task = {}
for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    groups = task_df[group_col].astype(str).to_numpy()
    splitter = GroupKFold(n_splits=RIDGE_OUTER_SPLITS)
    task_fold = np.full(len(task_df), -1, dtype=int)
    for fold, (_, test_idx) in enumerate(splitter.split(task_df, task_df[target_col], groups)):
        task_fold[test_idx] = fold
        for idx in test_idx:
            outer_assignment_rows.append({
                "sample_id": task_df.loc[idx, "sample_id"],
                "task": task,
                "outer_fold": int(fold),
                "borough_code": task_df.loc[idx, group_col],
                "target": float(task_df.loc[idx, target_col]),
            })
    assert (task_fold >= 0).all()
    fold_index_by_task[task] = task_fold

outer_folds = pd.DataFrame(outer_assignment_rows).sort_values(
    ["task", "sample_id"], kind="mergesort"
).reset_index(drop=True)
saved_outer_folds = pd.read_csv(RIDGE_OUTER_FOLDS_PATH).sort_values(
    ["task", "sample_id"], kind="mergesort"
).reset_index(drop=True)
pd.testing.assert_frame_equal(
    saved_outer_folds[outer_folds.columns], outer_folds,
    check_dtype=False, check_exact=False, rtol=0, atol=1e-12,
)
assert outer_folds.groupby(["task", "borough_code"])["outer_fold"].nunique().eq(1).all()

fold_assignment_hash = hashlib.sha256(
    pd.util.hash_pandas_object(outer_folds, index=False).values.tobytes()
).hexdigest()
assert fold_assignment_hash == source_06_spec["outer_fold_assignment_sha256"]
assert fold_assignment_hash == source_07_spec["outer_fold_assignment_sha256"]

model_spec_payload = []
for row in model_specs.itertuples(index=False):
    model_spec_payload.append({
        "task": row.task,
        "model_id": row.model_id,
        "base_feature_set": row.base_feature_set,
        "analysis_role": row.analysis_role,
        "n_features_manifest": int(row.n_features_manifest),
        "feature_columns_sha256": columns_sha256(model_features[row.model_id]),
    })

run_spec = {
    "run_spec_version": "09-v1-2026-08-23",
    "notebook": "09_targeted_dinov2_satclip_combination.ipynb",
    "question": "Does frozen DINOv2 plus frozen SatCLIP improve out-of-borough prediction relative to either component alone?",
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "outer_fold_assignment_sha256": fold_assignment_hash,
    "source_06_run_spec_sha256": source_06_spec_sha256,
    "source_07_run_spec_sha256": source_07_spec_sha256,
    "sklearn_version": sklearn.__version__,
    "target_column": target_col,
    "group_column": group_col,
    "outer_splits": int(RIDGE_OUTER_SPLITS),
    "inner_splits": int(RIDGE_INNER_SPLITS),
    "alpha_grid": [float(x) for x in RIDGE_ALPHA_GRID],
    "inner_selection_metric": "RMSE",
    "model_specifications": model_spec_payload,
    "comparators": comparator_specs.to_dict("records"),
    "inference": "descriptive fold-paired deltas, SD and wins; no independent-fold p-values",
    "dinov3_status": "out of scope by time-prioritisation decision",
    "preprocessing": {
        "numeric_imputation": "training-fold median",
        "numeric_scaling": "training-fold StandardScaler",
        "categorical_imputation": "training-fold constant __MISSING__",
        "categorical_encoding": "training-fold OneHotEncoder(handle_unknown=ignore)",
    },
}
run_spec_sha256 = hashlib.sha256(
    json.dumps(run_spec, sort_keys=True).encode("utf-8")
).hexdigest()

if IMAGE_LOCATION_RUN_SPEC_PATH.exists():
    with open(IMAGE_LOCATION_RUN_SPEC_PATH, "r") as f:
        existing_spec = json.load(f)
    assert existing_spec == run_spec, (
        "Existing Notebook-09 checkpoints belong to another run specification. "
        "Archive them before a documented rerun."
    )
else:
    atomic_json(run_spec, IMAGE_LOCATION_RUN_SPEC_PATH)

expected_runs_df = pd.DataFrame([
    {"task": row.task, "model_id": row.model_id, "outer_fold": fold}
    for row in model_specs.itertuples(index=False)
    for fold in range(RIDGE_OUTER_SPLITS)
])
expected_keys = set(map(tuple, expected_runs_df[["task", "model_id", "outer_fold"]].to_numpy()))
expected_prediction_rows = int(sum(
    task_counts[row.task] for row in model_specs.itertuples(index=False)
))
expected_paired_rows = int(len(comparator_specs) * RIDGE_OUTER_SPLITS)

assert len(expected_keys) == 25
assert expected_prediction_rows == 73194
assert expected_paired_rows == 75

print("Verified borough-fold SHA256:", fold_assignment_hash)
print("Notebook-09 run-spec SHA256:", run_spec_sha256)
print("Expected runs:", len(expected_keys))
print("Expected prediction rows:", expected_prediction_rows)
print("Expected paired rows:", expected_paired_rows)

## 5. Fit the five combined models

Each model is fitted in all five borough rounds. Its held-out predictions and performance measures are saved after every completed round.

In [ ]:
# Fit the five combined models and save round-level predictions.
if IMAGE_LOCATION_RESULTS_PATH.exists():
    completed = pd.read_csv(IMAGE_LOCATION_RESULTS_PATH)
    required = {"task", "model_id", "outer_fold", "run_spec_sha256"}
    assert required.issubset(completed.columns)
    assert not completed.duplicated(["task", "model_id", "outer_fold"]).any()
    assert completed["run_spec_sha256"].eq(run_spec_sha256).all()
    completed["outer_fold"] = completed["outer_fold"].astype(int)
    completed_keys = set(map(tuple, completed[["task", "model_id", "outer_fold"]].to_numpy()))
    assert completed_keys.issubset(expected_keys)
else:
    completed = pd.DataFrame()

result_rows = [] if completed.empty else completed.to_dict("records")

def checkpoint_is_valid(path, task, model_id, outer_fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            "sample_id", "task", "model_id", "outer_fold", "borough_code",
            "y_true", "y_pred", "run_spec_sha256",
        }
        if not required.issubset(p.columns) or p["sample_id"].duplicated().any():
            return False
        if not p["task"].eq(task).all() or not p["model_id"].eq(model_id).all():
            return False
        if not p["outer_fold"].astype(int).eq(outer_fold).all():
            return False
        if not p["run_spec_sha256"].eq(run_spec_sha256).all():
            return False
        if not np.isfinite(p["y_true"]).all() or not np.isfinite(p["y_pred"]).all():
            return False
        return set(p["sample_id"].astype(str)) == set(pd.Series(expected_ids).astype(str))
    except Exception:
        return False

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    y = pd.to_numeric(task_df[target_col], errors="raise").to_numpy()
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = fold_index_by_task[task]

    for spec_row in model_specs[model_specs["task"] == task].itertuples(index=False):
        model_id = spec_row.model_id
        cols = model_features[model_id]
        X = task_df[cols]

        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(task_fold == outer_fold)
            train_idx = np.flatnonzero(task_fold != outer_fold)
            run_key = (task, model_id, outer_fold)
            pred_file = IMAGE_LOCATION_CHUNK_DIR / f"{task}__{model_id}__fold{outer_fold}.parquet"

            completed_keys_now = {
                (r["task"], r["model_id"], int(r["outer_fold"])) for r in result_rows
            }
            checkpoint_ok = checkpoint_is_valid(
                pred_file, task, model_id, outer_fold,
                task_df.iloc[test_idx]["sample_id"].to_numpy(),
            )
            if run_key in completed_keys_now and checkpoint_ok:
                print("SKIP validated checkpoint:", run_key)
                continue

            print("\n" + "=" * 100)
            print(task, "|", model_id, "| outer fold", outer_fold)
            print("=" * 100)

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            g_train = groups[train_idx]
            assert set(g_train).isdisjoint(set(groups[test_idx]))

            inner = GroupKFold(n_splits=RIDGE_INNER_SPLITS)
            inner_splits = list(inner.split(X_train, y_train, g_train))
            for inner_train, inner_valid in inner_splits:
                assert set(g_train[inner_train]).isdisjoint(set(g_train[inner_valid]))

            pipe = build_pipeline(cols)
            search = GridSearchCV(
                estimator=pipe,
                param_grid={"ridge__alpha": RIDGE_ALPHA_GRID},
                scoring="neg_root_mean_squared_error",
                cv=inner_splits,
                refit=True,
                n_jobs=1,
                return_train_score=False,
                error_score="raise",
            )

            t0 = time.time()
            search.fit(X_train, y_train)
            elapsed_s = time.time() - t0
            pred = search.predict(X_test)
            assert len(pred) == len(test_idx) and np.isfinite(pred).all()

            best_alpha = float(search.best_params_["ridge__alpha"])
            alpha_edge = best_alpha in {float(min(RIDGE_ALPHA_GRID)), float(max(RIDGE_ALPHA_GRID))}
            result_row = {
                "task": task,
                "model_id": model_id,
                "base_feature_set": spec_row.base_feature_set,
                "analysis_role": spec_row.analysis_role,
                "outer_fold": int(outer_fold),
                "n_features_manifest": int(len(cols)),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
                "n_train_boroughs": int(len(np.unique(g_train))),
                "n_test_boroughs": int(len(np.unique(groups[test_idx]))),
                "best_alpha": best_alpha,
                "alpha_grid_edge": bool(alpha_edge),
                "inner_best_rmse": float(-search.best_score_),
                "r2": float(r2_score(y_test, pred)),
                "rmse": rmse(y_test, pred),
                "mae": float(mean_absolute_error(y_test, pred)),
                "fit_seconds": float(elapsed_s),
                "run_spec_sha256": run_spec_sha256,
            }

            pred_frame = pd.DataFrame({
                "sample_id": task_df.iloc[test_idx]["sample_id"].to_numpy(),
                "task": task,
                "model_id": model_id,
                "outer_fold": int(outer_fold),
                "borough_code": groups[test_idx],
                "y_true": y_test,
                "y_pred": pred,
                "run_spec_sha256": run_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)

            result_rows = [
                r for r in result_rows
                if (r["task"], r["model_id"], int(r["outer_fold"])) != run_key
            ]
            result_rows.append(result_row)
            atomic_csv(
                pd.DataFrame(result_rows).sort_values(
                    ["task", "model_id", "outer_fold"], kind="mergesort"
                ),
                IMAGE_LOCATION_RESULTS_PATH,
            )
            print(result_row)

            del search, pipe, X_train, X_test, pred, pred_frame
            gc.collect()

print("Notebook-09 fits are complete or safely checkpointed.")

## 6. Assemble the complete prediction set

The final summary requires all 25 model rounds and their exact held-out predictions. Task, model, borough, outcome and sample identity are verified before aggregation.

In [ ]:
# Verify the complete set of new predictions.
results = pd.read_csv(IMAGE_LOCATION_RESULTS_PATH)
results["outer_fold"] = results["outer_fold"].astype(int)
assert not results.duplicated(["task", "model_id", "outer_fold"]).any()
assert results["run_spec_sha256"].eq(run_spec_sha256).all()
actual_keys = set(map(tuple, results[["task", "model_id", "outer_fold"]].to_numpy()))

missing_keys = sorted(expected_keys - actual_keys)
unexpected_keys = sorted(actual_keys - expected_keys)
print("Completed fold-runs:", len(actual_keys), "/", len(expected_keys))
print("Missing:", len(missing_keys), "Unexpected:", len(unexpected_keys))
assert not missing_keys, "Notebook 09 incomplete: rerun Section 5"
assert not unexpected_keys

pred_frames = []
chunk_audit_rows = []
for task, model_id, outer_fold in sorted(expected_keys):
    path = IMAGE_LOCATION_CHUNK_DIR / f"{task}__{model_id}__fold{outer_fold}.parquet"
    assert path.exists(), f"Missing chunk: {path.name}"
    p = pd.read_parquet(path)
    required = {
        "sample_id", "task", "model_id", "outer_fold", "borough_code",
        "y_true", "y_pred", "run_spec_sha256",
    }
    assert required.issubset(p.columns)
    assert p["sample_id"].is_unique
    assert p["task"].eq(task).all() and p["model_id"].eq(model_id).all()
    assert p["outer_fold"].astype(int).eq(outer_fold).all()
    assert p["run_spec_sha256"].eq(run_spec_sha256).all()
    assert np.isfinite(p["y_true"]).all() and np.isfinite(p["y_pred"]).all()

    expected_fold = outer_folds[
        (outer_folds["task"] == task) & (outer_folds["outer_fold"] == outer_fold)
    ][["sample_id", "borough_code", "target"]].sort_values("sample_id").reset_index(drop=True)
    observed_fold = p[["sample_id", "borough_code", "y_true"]].sort_values(
        "sample_id"
    ).reset_index(drop=True)
    assert expected_fold["sample_id"].equals(observed_fold["sample_id"])
    assert expected_fold["borough_code"].astype(str).equals(observed_fold["borough_code"].astype(str))
    assert np.allclose(expected_fold["target"], observed_fold["y_true"], rtol=0, atol=1e-12)

    pred_frames.append(p)
    chunk_audit_rows.append({
        "task": task, "model_id": model_id, "outer_fold": int(outer_fold),
        "n_rows": int(len(p)), "file": path.name,
    })

preds = pd.concat(pred_frames, ignore_index=True)
assert not preds.duplicated(["sample_id", "task", "model_id", "outer_fold"]).any()
assert len(preds) == expected_prediction_rows

print("Validated chunks:", len(chunk_audit_rows))
print("Validated prediction rows:", len(preds))

## 7. Calculate within-round differences

The DINOv2-only, SatCLIP-only and all-representation comparators use the same held-out boroughs. A positive change in R² and a negative change in RMSE or MAE favour the DINOv2–SatCLIP pair.

In [ ]:
# Load matching DINOv2, SatCLIP and full-fusion results for within-round comparison.
source_06_results = pd.read_csv(RIDGE_CORE_RESULTS_PATH)
source_07_results = pd.read_csv(INCREMENTAL_RESULTS_PATH)
source_06_preds = pd.read_parquet(RIDGE_CORE_PREDICTIONS_PATH)
source_07_preds = pd.read_parquet(INCREMENTAL_PREDICTIONS_PATH)

assert len(source_06_results) == 135
assert len(source_07_results) == 205
assert not source_06_results.duplicated(["task", "feature_set", "outer_fold"]).any()
assert not source_07_results.duplicated(["task", "model_id", "outer_fold"]).any()
assert len(source_06_preds) == 365761
assert len(source_07_preds) == 645761

paired_rows = []
for comp in comparator_specs.itertuples(index=False):
    new_g = results[
        (results["task"] == comp.task) & (results["model_id"] == comp.new_model_id)
    ][["outer_fold", "r2", "rmse", "mae"]].copy()

    if comp.source_notebook == "06":
        source_g = source_06_results[
            (source_06_results["task"] == comp.task)
            & (source_06_results["feature_set"] == comp.comparator_id)
        ][["outer_fold", "r2", "rmse", "mae"]].copy()
    else:
        source_g = source_07_results[
            (source_07_results["task"] == comp.task)
            & (source_07_results["model_id"] == comp.comparator_id)
        ][["outer_fold", "r2", "rmse", "mae"]].copy()

    paired = new_g.merge(
        source_g, on="outer_fold", how="inner", validate="one_to_one",
        suffixes=("_new", "_comparator"),
    )
    assert len(paired) == RIDGE_OUTER_SPLITS
    for row in paired.itertuples(index=False):
        paired_rows.append({
            "task": comp.task,
            "new_model_id": comp.new_model_id,
            "comparator_role": comp.comparator_role,
            "source_notebook": comp.source_notebook,
            "comparator_id": comp.comparator_id,
            "outer_fold": int(row.outer_fold),
            "new_r2": float(row.r2_new),
            "comparator_r2": float(row.r2_comparator),
            "delta_r2": float(row.r2_new - row.r2_comparator),
            "new_rmse": float(row.rmse_new),
            "comparator_rmse": float(row.rmse_comparator),
            "delta_rmse": float(row.rmse_new - row.rmse_comparator),
            "new_mae": float(row.mae_new),
            "comparator_mae": float(row.mae_comparator),
            "delta_mae": float(row.mae_new - row.mae_comparator),
        })

paired = pd.DataFrame(paired_rows).sort_values(
    ["task", "new_model_id", "comparator_role", "outer_fold"], kind="mergesort"
)
assert len(paired) == expected_paired_rows
assert paired.groupby(["task", "new_model_id", "comparator_role"]).size().eq(5).all()

display(paired.groupby(["task", "new_model_id", "comparator_role"])[
    ["delta_r2", "delta_rmse", "delta_mae"]
].mean())

## 8. Summarise performance and consistency

Absolute performance is reported alongside the average within-round difference and the number of rounds in which the pair improves. A very small increase in a grand mean is not interpreted as meaningful unless error measures and round-level directions support it.

In [ ]:
# Summarise absolute performance and the size and consistency of any gain.
summary_rows = []
for (task, model_id), g in results.groupby(["task", "model_id"]):
    pred_g = preds[(preds["task"] == task) & (preds["model_id"] == model_id)]
    spec = model_specs[model_specs["model_id"] == model_id].iloc[0]
    assert len(g) == 5 and len(pred_g) == task_counts[task]
    summary_rows.append({
        "task": task,
        "model_id": model_id,
        "base_feature_set": spec["base_feature_set"],
        "analysis_role": spec["analysis_role"],
        "n_features": int(spec["n_features_manifest"]),
        "mean_r2": float(g["r2"].mean()),
        "sd_r2": float(g["r2"].std(ddof=1)),
        "mean_rmse": float(g["rmse"].mean()),
        "sd_rmse": float(g["rmse"].std(ddof=1)),
        "mean_mae": float(g["mae"].mean()),
        "sd_mae": float(g["mae"].std(ddof=1)),
        "pooled_r2": float(r2_score(pred_g["y_true"], pred_g["y_pred"])),
        "pooled_rmse": rmse(pred_g["y_true"], pred_g["y_pred"]),
        "pooled_mae": float(mean_absolute_error(pred_g["y_true"], pred_g["y_pred"])),
        "alpha_edge_hits": int(coerce_bool(g["alpha_grid_edge"]).sum()),
        "median_best_alpha": float(g["best_alpha"].median()),
        "total_fit_minutes": float(g["fit_seconds"].sum() / 60),
    })

summary = pd.DataFrame(summary_rows).sort_values(["task", "model_id"], kind="mergesort")

comparison_summary = paired.groupby(
    ["task", "new_model_id", "comparator_role", "source_notebook", "comparator_id"],
    as_index=False,
).agg(
    mean_delta_r2=("delta_r2", "mean"),
    sd_delta_r2=("delta_r2", "std"),
    r2_wins=("delta_r2", lambda s: int((s > 0).sum())),
    mean_delta_rmse=("delta_rmse", "mean"),
    sd_delta_rmse=("delta_rmse", "std"),
    rmse_wins=("delta_rmse", lambda s: int((s < 0).sum())),
    mean_delta_mae=("delta_mae", "mean"),
    sd_delta_mae=("delta_mae", "std"),
    mae_wins=("delta_mae", lambda s: int((s < 0).sum())),
)
comparison_summary["strict_all_metric_win"] = (
    (comparison_summary["mean_delta_r2"] > 0)
    & (comparison_summary["mean_delta_rmse"] < 0)
    & (comparison_summary["mean_delta_mae"] < 0)
)

display(summary)
display(comparison_summary)

## 9. Finalise the results

All required rows, predictions and comparator identities are confirmed, and the selected Ridge penalties are checked against the candidate-range boundaries. A null or mixed comparison is retained as a substantive finding rather than treated as an unsuccessful run.

In [ ]:
# Validate penalty selection and save the completed comparison.
edge_counts = results.assign(
    alpha_grid_edge_bool=coerce_bool(results["alpha_grid_edge"])
).groupby(["task", "model_id"])["alpha_grid_edge_bool"].sum().reset_index(name="edge_hits")
repeated_edge = edge_counts[edge_counts["edge_hits"] >= 3]

integrity_gate_pass = bool(
    len(actual_keys) == len(expected_keys)
    and len(preds) == expected_prediction_rows
    and len(paired) == expected_paired_rows
    and source_06_audit["integrity_gate_pass"]
    and source_07_audit["integrity_gate_pass"]
)
interpretation_gate_pass = bool(
    integrity_gate_pass
    and source_06_audit["interpretation_gate_pass"]
    and source_07_audit["interpretation_gate_pass"]
    and repeated_edge.empty
)

audit_summary = {
    "run_spec_path": str(IMAGE_LOCATION_RUN_SPEC_PATH),
    "run_spec_sha256": run_spec_sha256,
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "outer_fold_assignment_sha256": fold_assignment_hash,
    "source_06_run_spec_sha256": source_06_spec_sha256,
    "source_07_run_spec_sha256": source_07_spec_sha256,
    "source_06_integrity_gate_pass": bool(source_06_audit["integrity_gate_pass"]),
    "source_06_interpretation_gate_pass": bool(source_06_audit["interpretation_gate_pass"]),
    "source_07_integrity_gate_pass": bool(source_07_audit["integrity_gate_pass"]),
    "source_07_interpretation_gate_pass": bool(source_07_audit["interpretation_gate_pass"]),
    "new_models": int(len(model_specs)),
    "expected_fold_runs": int(len(expected_keys)),
    "completed_fold_runs": int(len(actual_keys)),
    "expected_prediction_rows": int(expected_prediction_rows),
    "actual_prediction_rows": int(len(preds)),
    "prediction_chunks_validated": int(len(chunk_audit_rows)),
    "expected_paired_rows": int(expected_paired_rows),
    "actual_paired_rows": int(len(paired)),
    "comparison_groups": int(len(comparison_summary)),
    "n_alpha_grid_edge_hits": int(coerce_bool(results["alpha_grid_edge"]).sum()),
    "n_models_with_repeated_edge_hits": int(len(repeated_edge)),
    "alpha_grid_adequacy_pass": bool(repeated_edge.empty),
    "primary_question_comparators": ["DINOv2_only", "SatCLIP_only"],
    "full_fusion_role": "secondary compactness benchmark",
    "inference": "descriptive fold-paired deltas, SD and win counts; no independent-fold p-values",
    "dinov3_status": "out of scope",
    "integrity_gate_pass": integrity_gate_pass,
    "interpretation_gate_pass": interpretation_gate_pass,
}

assert integrity_gate_pass, "Notebook-09 integrity gate failed"
assert interpretation_gate_pass, (
    "Interpretation paused: inspect repeated alpha-grid edge selections before substantive claims"
)

atomic_csv(results.sort_values(["task", "model_id", "outer_fold"]), IMAGE_LOCATION_RESULTS_PATH)
atomic_parquet(preds.sort_values(["task", "model_id", "sample_id"]), IMAGE_LOCATION_PREDICTIONS_PATH)
atomic_csv(summary, IMAGE_LOCATION_SUMMARY_PATH)
atomic_csv(paired, IMAGE_LOCATION_DELTAS_PATH)
atomic_csv(comparison_summary, IMAGE_LOCATION_COMPARISON_SUMMARY_PATH)
atomic_json(audit_summary, IMAGE_LOCATION_AUDIT_PATH)

print(json.dumps(audit_summary, indent=2))
print("\nNotebook 09: INTEGRITY PASS + INTERPRETATION PASS")

## Interpretation

DINOv2 supplies almost all of the useful information in this particular image–location pair. SatCLIP alone is weak for EPC and offers only a small, metric-dependent change for PTAL. The result supports reporting DINOv2 as the stronger component and the all-representation model as the better-performing fusion, while noting that simply joining aerial and location embeddings does not guarantee complementary value.